# Demo 01: Core CUDA Python APIs

**Learning objectives:**
- Understand CUDA Python's device management model
- Learn how to compile and launch custom kernels from Python
- Explore GPU performance parameters analytically using the kernel model library
- Understand memory hierarchy: global, shared, pinned

> **Note:** Run cells top-to-bottom. Restarting the kernel resets all state.
> GPU cells are guarded with `if GPU_AVAILABLE:` and are skipped automatically
> on CPU-only machines.

In [ ]:
# GPU detection — always runs
import subprocess
try:
    subprocess.run(["nvidia-smi"], check=True, capture_output=True)
    GPU_AVAILABLE = True
    print("GPU detected — all cells will run.")
except (FileNotFoundError, subprocess.CalledProcessError):
    GPU_AVAILABLE = False
    print("No GPU detected — GPU cells will be skipped (CPU demonstrations still work.)")

## Device Specifications

The `DeviceSpec` class provides hardware parameters for any supported GPU,
even without a physical device present. This is useful for understanding
theoretical performance limits before writing any kernel code.

Supported GPU names: `v100`, `a100`, `a100-40gb`, `h100`, `b100`,
`rtx3090`, `rtx4090`, `rtx5090`.

In [ ]:
# GPU-free — always runs
import sys
sys.path.insert(0, "..")

from src.kernel_model import DeviceSpec, OccupancyModel, RooflineModel

# Explore specs for several GPU generations
print(f"{'GPU':<20} | {'SMs':>4} | {'Peak FP32 (GFLOP/s)':>20} | {'BW (GB/s)':>10} | sm_version")
print("-" * 75)
for gpu_name in ["v100", "a100", "h100", "rtx4090"]:
    spec = DeviceSpec.from_name(gpu_name)
    print(f"{spec.name:<20} | {spec.sm_count:>4} | "
          f"{spec.peak_fp32_gflops:>20,.0f} | "
          f"{spec.memory_bandwidth_gbs:>10,.0f} | "
          f"{spec.sm_version}")

## Occupancy Analysis

**Occupancy** measures how many warps are in flight on an SM at once.
Higher occupancy helps hide memory latency by keeping execution units busy
while other warps wait for data.

The occupancy model computes the **binding constraint** — the resource that
limits how many thread blocks fit on an SM simultaneously:

| Resource | Typical limiter for... |
|----------|------------------------|
| `warps` | small block sizes |
| `shared_memory` | kernels using large shared memory tiles |
| `registers` | register-heavy kernels (e.g. unrolled loops) |
| `blocks` | architectures with low max-blocks-per-SM |

**Try:** Change `registers_per_thread` to 32 or 64 and observe how the
limiter shifts from `warps` to `registers`.

In [ ]:
# GPU-free — always runs
# User-editable parameters
GPU_NAME = "a100"           # Try: "v100", "h100", "rtx4090"
SHARED_MEM_PER_BLOCK = 0    # bytes; 0 = no shared memory used
REGISTERS_PER_THREAD = 16   # Try: 8, 16, 32, 64

device = DeviceSpec.from_name(GPU_NAME)
model = OccupancyModel(device)

print(f"Occupancy sweep for a vector-add kernel on {device.name}")
print(f"(shared_mem_per_block={SHARED_MEM_PER_BLOCK} B, registers_per_thread={REGISTERS_PER_THREAD})\n")

results = model.sweep_block_sizes(
    shared_mem_per_block=SHARED_MEM_PER_BLOCK,
    registers_per_thread=REGISTERS_PER_THREAD,
)

print(f"{'Block Size':>10}  {'Warps/Block':>11}  {'Active Blocks/SM':>16}  {'Occupancy':>9}  {'Limiter'}")
print("-" * 65)
for r in results:
    print(f"{r.block_size:>10}  {r.warps_per_block:>11}  {r.active_blocks_per_sm:>16}  "
          f"{r.occupancy * 100:>8.1f}%  {r.limiting_resource}")

best = max(results, key=lambda r: r.occupancy)
print(f"\nRecommended block size: {best.block_size} ({best.occupancy * 100:.1f}% occupancy)")

## Roofline Analysis

The **roofline model** classifies a kernel as *compute-bound* or *memory-bound*
based on its **arithmetic intensity** (FLOP per byte of memory traffic).

```
arithmetic_intensity = total_FLOPs / total_bytes_accessed

ridge_point = peak_FLOP/s / peak_BW (GB/s)

if arithmetic_intensity < ridge_point:
    bound = "memory"   # roofline ceiling is BW × intensity
else:
    bound = "compute"  # roofline ceiling is peak FLOP/s
```

**Vector-add** performs one addition per element with 2 reads + 1 write,
giving a very low arithmetic intensity — it is always memory-bound.

In [ ]:
# GPU-free — always runs
# User-editable parameters
N = 1_000_000   # vector length

flops = float(N)              # one add per element
bytes_accessed = 3 * N * 4   # 2 reads + 1 write of float32

a100 = DeviceSpec.from_name("a100")
roof_model = RooflineModel(a100)
result = roof_model.compute(flops=flops, bytes_accessed=bytes_accessed)

print(f"Vector-add kernel (N={N:,}, float32) on {a100.name}:")
print(f"  Arithmetic intensity : {result.arithmetic_intensity:.4f} FLOP/byte")
print(f"  Ridge point          : {result.ridge_point:.2f} FLOP/byte")
print(f"  Bound                : {result.bound}")
print(f"  Predicted throughput : {result.predicted_gflops:,.1f} GFLOP/s")
print(f"  Peak FP32            : {a100.peak_fp32_gflops:,.0f} GFLOP/s")

util_pct = result.predicted_gflops / a100.peak_fp32_gflops * 100
print(f"  FP32 utilisation     : {util_pct:.1f}% of peak compute")

## GPU Generation Comparison

How does the ridge point vary across GPU generations?
Higher ridge points mean a wider range of kernels become memory-bound.

In [ ]:
# GPU-free — always runs
print(f"{'GPU':<20}  {'Peak FP32 (GFLOP/s)':>20}  {'BW (GB/s)':>10}  {'Ridge (FLOP/B)':>14}")
print("-" * 72)
for gpu_name in ["v100", "a100", "h100", "rtx4090"]:
    spec = DeviceSpec.from_name(gpu_name)
    ridge = spec.peak_fp32_gflops / spec.memory_bandwidth_gbs
    print(f"{spec.name:<20}  {spec.peak_fp32_gflops:>20,.0f}  "
          f"{spec.memory_bandwidth_gbs:>10,.0f}  {ridge:>14.1f}")

print("\nVector-add arithmetic intensity: ~0.083 FLOP/byte — always memory-bound.")

## Live GPU Demo

The cell below requires a physical NVIDIA GPU with CUDA 12.x installed.
It is skipped automatically on CPU-only machines.

In [ ]:
# REQUIRES GPU
if GPU_AVAILABLE:
    print("GPU available! Run the full live demo from the repo root:")
    print("  python demos/01_core_apis/main.py")
    print("")
    print("The demo covers:")
    print("  - Device info (SM count, VRAM, compute capability)")
    print("  - Vector-add kernel: compile PTX, launch, verify vs NumPy")
    print("  - Pinned vs pageable memory transfer benchmark")
    print("  - Load data from disk/URL directly to GPU memory")
else:
    print("No GPU detected.")
    print("")
    print("To run the GPU-accelerated demo, install CUDA Toolkit 12.x and:")
    print("  pip install cuda-python")
    print("  python demos/01_core_apis/main.py")

## Summary

In this notebook we explored:

1. **GPU device specifications** — comparing hardware parameters (SM count, peak FP32
   throughput, memory bandwidth) across GPU generations from V100 to RTX 4090
2. **Occupancy analysis** — how block size and shared memory affect warp utilization,
   and which resource is the binding constraint
3. **Roofline model** — classifying kernels as compute-bound or memory-bound based on
   arithmetic intensity; vector-add is always memory-bound at ~0.083 FLOP/byte

All analysis ran in pure Python without a GPU.

**Next steps:**
- Modify `REGISTERS_PER_THREAD` in the occupancy cell and observe how the
  limiter shifts from `warps` to `registers`
- Try `GPU_NAME = "h100"` to see how H100's higher memory bandwidth
  changes the roofline ridge point
- Proceed to `02_kmeans.ipynb` for the first GPU ML algorithm demo
- For live kernel execution, run `python demos/01_core_apis/main.py` with
  a CUDA 12.x GPU available